In [1]:
import pandas as pd

df1 = pd.read_csv('../datasets/Crimes_-_2001_to_Present.csv', low_memory=False)
df2 = pd.read_csv('../datasets/chicago crimes.csv', low_memory=False)

df_crimes = pd.concat([df1, df2], ignore_index=True)

print("Total rows after merge:", len(df_crimes))
print("Columns:", df_crimes.columns.tolist())

Total rows after merge: 16285565
Columns: ['ID', 'Case Number', 'Date', 'Block', 'IUCR', 'Primary Type', 'Description', 'Location Description', 'Arrest', 'Domestic', 'Beat', 'District', 'Ward', 'Community Area', 'FBI Code', 'X Coordinate', 'Y Coordinate', 'Year', 'Updated On', 'Latitude', 'Longitude', 'Location']


In [2]:
cols = ['ID', 'Date', 'Primary Type', 'Description',
        'Location Description', 'Arrest', 'Domestic',
        'Beat', 'District', 'Ward', 'Community Area',
        'Latitude', 'Longitude', 'Year']

df_crimes = df_crimes[cols]

df_crimes.columns = ['crime_id', 'date', 'crime_type', 'description',
                     'location_desc', 'arrest', 'domestic',
                     'beat', 'district', 'ward', 'community_area',
                     'latitude', 'longitude', 'year']

print("Columns renamed successfully")
print(df_crimes.head(3))

Columns renamed successfully
   crime_id                    date crime_type              description  \
0  10224738  09/05/2015 01:30:00 PM    BATTERY  DOMESTIC BATTERY SIMPLE   
1  10224739  09/04/2015 11:30:00 AM      THEFT           POCKET-PICKING   
2  11646166  09/01/2018 12:01:00 AM      THEFT                OVER $500   

  location_desc  arrest  domestic  beat  district  ward  community_area  \
0     RESIDENCE   False      True   924       9.0  12.0            61.0   
1       CTA BUS   False     False  1511      15.0  29.0            25.0   
2     RESIDENCE   False      True   631       6.0   8.0            44.0   

    latitude  longitude  year  
0  41.815117   -87.6700  2015  
1  41.895080   -87.7654  2015  
2        NaN        NaN  2018  


In [3]:
df_crimes['date'] = pd.to_datetime(df_crimes['date'], errors='coerce')

df_crimes['hour']        = df_crimes['date'].dt.hour
df_crimes['month']       = df_crimes['date'].dt.month
df_crimes['day_of_week'] = df_crimes['date'].dt.day_name()
df_crimes['is_weekend']  = df_crimes['day_of_week'].isin(['Saturday','Sunday']).astype(int)

print("Date features extracted")
print(df_crimes[['date','hour','month','day_of_week','is_weekend']].head(5))

C:\Users\ASUS\AppData\Local\Temp\ipykernel_33508\3299169488.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_crimes['date'] = pd.to_datetime(df_crimes['date'], errors='coerce')


Date features extracted
                 date  hour  month day_of_week  is_weekend
0 2015-09-05 13:30:00    13      9    Saturday           1
1 2015-09-04 11:30:00    11      9      Friday           0
2 2018-09-01 00:01:00     0      9    Saturday           1
3 2015-09-05 12:45:00    12      9    Saturday           1
4 2015-09-05 13:00:00    13      9    Saturday           1


In [4]:
before = len(df_crimes)

# Drop missing locations
df_crimes.dropna(subset=['latitude', 'longitude'], inplace=True)

# Drop duplicate crime IDs
df_crimes.drop_duplicates(subset='crime_id', inplace=True)

# Convert True/False to 1/0
df_crimes['arrest']   = df_crimes['arrest'].astype(str).str.strip().str.upper().map({'TRUE':1,'FALSE':0}).fillna(0).astype(int)
df_crimes['domestic'] = df_crimes['domestic'].astype(str).str.strip().str.upper().map({'TRUE':1,'FALSE':0}).fillna(0).astype(int)

after = len(df_crimes)
print(f"Rows before: {before} → after cleaning: {after}")
print(f"Removed: {before - after} bad rows")

Rows before: 16285565 → after cleaning: 8408058
Removed: 7877507 bad rows


In [5]:
df_temp = pd.read_csv('../datasets/temperature.csv')

# Temperature is in Kelvin — convert to Celsius
# Keep only Chicago column
df_temp = df_temp[['datetime', 'Chicago']]
df_temp.columns = ['date', 'temp_celsius']

# Convert Kelvin to Celsius
df_temp['temp_celsius'] = df_temp['temp_celsius'] - 273.15

df_temp['date'] = pd.to_datetime(df_temp['date']).dt.date
df_temp['temp_celsius'] = pd.to_numeric(df_temp['temp_celsius'], errors='coerce')
df_temp.dropna(inplace=True)

print("Temperature sample:")
print(df_temp.head(3))

# Join to crimes by date
df_crimes['date_only'] = df_crimes['date'].dt.date
df_crimes = df_crimes.merge(df_temp, left_on='date_only', right_on='date', how='left')
df_crimes.drop(columns=['date_y', 'date_only'], inplace=True)
df_crimes.rename(columns={'date_x': 'date'}, inplace=True)

print("\nAfter weather join:", df_crimes.shape)
print(df_crimes[['date','crime_type','temp_celsius']].head(3))

Temperature sample:
         date  temp_celsius
1  2012-10-01     10.860000
2  2012-10-01     10.904691
3  2012-10-01     11.027412



After weather join: (41247443, 19)
                 date crime_type  temp_celsius
0 2015-09-05 13:30:00    BATTERY         23.89
1 2015-09-05 13:30:00    BATTERY         22.87
2 2015-09-05 13:30:00    BATTERY         22.28


In [6]:
df_la = pd.read_csv('../datasets/Crime_Data_from_2020_to_Present.csv', low_memory=False)

# Keep needed columns
df_la = df_la[['DR_NO', 'DATE OCC', 'Crm Cd Desc', 'LAT', 'LON', 'AREA NAME']]
df_la.columns = ['crime_id', 'date', 'crime_type', 'latitude', 'longitude', 'district']

df_la['date'] = pd.to_datetime(df_la['date'], errors='coerce')
df_la.dropna(subset=['latitude', 'longitude'], inplace=True)
df_la.drop_duplicates(subset='crime_id', inplace=True)

# Remove invalid coordinates
df_la = df_la[(df_la['latitude'] != 0) & (df_la['longitude'] != 0)]

print("LA clean rows:", len(df_la))
print(df_la.head(3))

LA clean rows: 754875
    crime_id       date                                 crime_type  latitude  \
0   10304468 2020-01-08                   BATTERY - SIMPLE ASSAULT   34.0141   
1  190101086 2020-01-01                   BATTERY - SIMPLE ASSAULT   34.0459   
2  200110444 2020-02-13  SEX OFFENDER REGISTRANT OUT OF COMPLIANCE   34.0448   

   longitude   district  
0  -118.2978  Southwest  
1  -118.2545    Central  
2  -118.2474    Central  


C:\Users\ASUS\AppData\Local\Temp\ipykernel_33508\4142720067.py:7: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_la['date'] = pd.to_datetime(df_la['date'], errors='coerce')


In [7]:
df_census = pd.read_csv('../datasets/acs2017_census_tract_data.csv')

# Keep useful columns only
df_census = df_census[['TractId', 'State', 'County', 'TotalPop',
                        'Unemployment', 'Poverty', 'Income']]
df_census.dropna(inplace=True)

print("Census clean rows:", len(df_census))
print(df_census.head(3))

Census clean rows: 72884
      TractId    State          County  TotalPop  Unemployment  Poverty  \
0  1001020100  Alabama  Autauga County      1845           4.6     10.7   
1  1001020200  Alabama  Autauga County      2172           3.4     22.4   
2  1001020300  Alabama  Autauga County      3385           4.7     14.7   

    Income  
0  67826.0  
1  41287.0  
2  46806.0  


In [8]:
df_crimes.to_csv('../datasets/chicago_clean.csv', index=False)
df_la.to_csv('../datasets/la_clean.csv', index=False)
df_census.to_csv('../datasets/census_clean.csv', index=False)

print("✅ All 3 cleaned files saved!")
print("Chicago rows:", len(df_crimes))
print("LA rows:     ", len(df_la))
print("Census rows: ", len(df_census))

✅ All 3 cleaned files saved!
Chicago rows: 41247443
LA rows:      754875
Census rows:  72884
